# Assignment 2, Hint 1: a bigger U-Net

CSCI 6379 · companion to the [Assignment 2 hints page](https://dongchul.kim/csci6379-fall2026/assignment/2/hints).

The starter's U-Net has three levels and `WIDTH = 16`: under half a million parameters, and its bottleneck works at 32 x 32. This version has **four levels** and **`WIDTH = 32`**: twice the channels at every level (more kinds of pattern it can tell apart) and a bottleneck at 16 x 16 (each of its cells summarises a larger part of the image, so the network sees more context). It has 7,763,041 parameters, under the 10 million limit. This is FIX 2 done in full.

**Where it goes.** `unet.py` in your notebook. Replace the `WIDTH = 16` line and the whole `class UNet`. Leave `block`, `to01`, `normalize` and `prep` as they are.

This notebook holds **only the code of this hint**. It is not a training program and it
does not touch the assignment data. Copy the marked cell into your own Assignment 2
notebook, change `RUN_NAME` there, and train as before. The last cell is a small check
you can run here to see the hint working.

## Setup for this notebook only (do not copy)

The imports and the starter's `block`, exactly as in your `unet.py`, so the cells below can run here.

In [ ]:
import torch
import torch.nn as nn


def block(i, o):
    # The starter's building block: two 3x3 convolutions, each followed by batch norm and ReLU.
    # padding=1 keeps the height and width, so only the pooling and up-sampling change the size.
    return nn.Sequential(
        nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
    )

## Hint 1: copy this into `unet.py`

In [ ]:
# ---- copy from here into unet.py: it replaces "WIDTH = 16" and the whole class UNet ----

WIDTH = 32    # channels at the top level (was 16). Each level down doubles them: 32, 64, 128, 256, then 512.


class UNet(nn.Module):
    # Four levels down, a bottleneck, four levels up (the starter had three of each).
    #
    #   size of the feature map     encoder (down)        decoder (up)
    #   256 x 256                    enc1:   3 -> 32       dec1: 64 -> 32    (+ head: 32 -> 1)
    #   128 x 128                    enc2:  32 -> 64       dec2: 128 -> 64
    #    64 x 64                     enc3:  64 -> 128      dec3: 256 -> 128
    #    32 x 32                     enc4: 128 -> 256      dec4: 512 -> 256   <- the new level
    #    16 x 16                     mid:  256 -> 512 (the bottleneck)        <- one level deeper
    #
    # Every decoder block takes twice its output channels as input: half come up from the
    # level below (through the transposed convolution), half come across from the encoder
    # at the same size (the skip connection, joined with torch.cat).
    def __init__(self, w=WIDTH):
        super().__init__()
        self.pool = nn.MaxPool2d(2)                  # halves the height and width on the way down

        # encoder: each level sees the image at half the size of the one above, with twice the channels
        self.enc1 = block(3, w)                      # 256 x 256, input is the RGB image (3 channels)
        self.enc2 = block(w, 2 * w)                  # 128 x 128
        self.enc3 = block(2 * w, 4 * w)              #  64 x 64
        self.enc4 = block(4 * w, 8 * w)              #  32 x 32   NEW: the fourth level
        self.mid  = block(8 * w, 16 * w)             #  16 x 16   the bottleneck, now one level deeper

        # decoder: a 2x2 transposed convolution with stride 2 doubles the height and width and
        # halves the channels; the block then mixes it with the skip from the encoder
        self.up4  = nn.ConvTranspose2d(16 * w, 8 * w, 2, stride=2)   # 16 -> 32     NEW
        self.dec4 = block(16 * w, 8 * w)                             # 8w up + 8w skip in   NEW
        self.up3  = nn.ConvTranspose2d(8 * w, 4 * w, 2, stride=2)    # 32 -> 64
        self.dec3 = block(8 * w, 4 * w)
        self.up2  = nn.ConvTranspose2d(4 * w, 2 * w, 2, stride=2)    # 64 -> 128
        self.dec2 = block(4 * w, 2 * w)
        self.up1  = nn.ConvTranspose2d(2 * w, w, 2, stride=2)        # 128 -> 256
        self.dec1 = block(2 * w, w)

        self.head = nn.Conv2d(w, 1, 1)               # 1x1 convolution: one logit per pixel

    def forward(self, x):
        # down: keep each level's output (e1 .. e4) for its skip connection
        e1 = self.enc1(x)                            # (N,  32, 256, 256)
        e2 = self.enc2(self.pool(e1))                # (N,  64, 128, 128)
        e3 = self.enc3(self.pool(e2))                # (N, 128,  64,  64)
        e4 = self.enc4(self.pool(e3))                # (N, 256,  32,  32)
        m  = self.mid(self.pool(e4))                 # (N, 512,  16,  16)

        # up: up-sample, put the skip from the same level next to it (dim=1 is channels), mix
        d4 = self.dec4(torch.cat([self.up4(m),  e4], dim=1))   # (N, 256,  32,  32)
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))   # (N, 128,  64,  64)
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))   # (N,  64, 128, 128)
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))   # (N,  32, 256, 256)
        return self.head(d1)                         # (N, 1, 256, 256) logits; sigmoid gives probabilities

# ---- copy up to here ----

## Check (run here)

The parameter count must be 7,763,041 and the output must be `(1, 1, 256, 256)`.

In [ ]:
net = UNet()
n_params = sum(p.numel() for p in net.parameters())
print(f"parameters: {n_params:,}  (limit 10,000,000)")
out = net(torch.zeros(1, 3, 256, 256))
print("output shape:", tuple(out.shape))
assert n_params == 7_763_041 and tuple(out.shape) == (1, 1, 256, 256)
print("OK. In your notebook: paste the cell into unet.py, change RUN_NAME, and rerun Step 5's check.")